# Experiment 5: Music Generation Using LSTM and MIDI Representation
\n**Deep Learning Lab** — notebook prepared from the supplied experiment notes. Update dataset paths before running.
\n## Procedure
1. Load the dataset.
2. Perform the required preprocessing/EDA.
3. Train the model and evaluate it.
\n## Dataset note
Required datasets are not bundled in this repository. See the README for setup reminders.


In [ ]:
%pip install pretty_midi tensorflow numpy
import glob
import numpy as np
import pretty_midi
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

notes = []
for file in glob.glob("midi_files/*.mid"):
    midi_file = pretty_midi.PrettyMIDI(file)
    for inst in midi_file.instruments:
        notes.extend(n.pitch for n in inst.notes)

if len(notes) <= 16:
    raise ValueError("Add MIDI files to midi_files/ with more than 16 note events.")
seq_len = 16
X, y = [], []
for i in range(len(notes) - seq_len):
    X.append(notes[i:i + seq_len])
    y.append(notes[i + seq_len])
X, y = np.array(X), np.array(y)

model = Sequential([Embedding(128, 32), LSTM(64), Dense(128, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.fit(X, y, epochs=10, verbose=0)
seq, generated = list(notes[:seq_len]), []
for _ in range(100):
    pred = model.predict(np.array([seq]), verbose=0)[0]
    pitch = np.random.choice(128, p=pred)
    generated.append(pitch)
    seq = seq[1:] + [pitch]

out = pretty_midi.PrettyMIDI()
instrument = pretty_midi.Instrument(program=0)
for i, pitch in enumerate(generated):
    instrument.notes.append(pretty_midi.Note(velocity=90, pitch=int(pitch), start=i*0.5, end=i*0.5+0.4))
out.instruments.append(instrument)
out.write("generated_midi.mid")
print("MIDI file generated")

## Result
Run the notebook and record the actual metrics and generated files here. Do not report unobserved results.
